# GF(corbel_small)

In [ ]:
from pathlib import Path
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["input", "output1", "output2"])
args = Args("Corbel2301_block2_June2019_crop_ali_crop.mrc",
            "Corbel2301_block2_June2019_crop_ali_crop__GF.mrc",
            "Corbel2301_block2_June2019_crop_ali_crop__GF.pdf")

In [ ]:
from my_google_auth import DriveHandler
service = DriveHandler.get_drive_service()
handler = DriveHandler.DriveHandler(service)

In [ ]:
DRIVE_TOMOGRADENOISING_TMP       = '1hGHvkP46fxLCQbUlyYhAS_eVl6PollQM'  # "tmp" folder
DRIVE_TOMOGRADENOISING_TOMOGRAMS = '1hfAOv6etLjB16K-nCrg-0u24iZBvmr1-'  # "Tomograms" folder

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        file_id = handler.find_file_id(drive_file_name=output, drive_folder_id=DRIVE_TOMOGRADENOISING_TMP)
        if file_id == None:
            print(f"{output} does not exist in Google Drive. Creating ...")
        else:
            print(f"Downloading {output} from Google Drive")
            success = handler.download(file_id, local_save_path=output)

In [ ]:
import logging
import numpy as np
import scipy.ndimage
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
#from ipywidgets import *
import cv2
import time
#import kernels
import skimage
#from skimage import io as skimage_io
import mrcfile
import logging
import information_theory
from denoising.volume.gaussian import Monochrome_Denoising as GF

In [ ]:
#file_path = Path("~/data/vols/" + args.input).expanduser()
file_path = (Path.home() / "data" / "vols" / args.input)
if file_path.exists():
    print(f"Found local {args.input}")
else:
    file_id = handler.find_file_id(drive_file_name=args.input, drive_folder_id=DRIVE_TOMOGRADENOISING_TOMOGRAMS)
    success = handler.download(file_id, local_save_path=file_path)

In [ ]:
logging.basicConfig(format="[%(filename)s:%(lineno)s %(funcName)s()] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [ ]:
#noisy = skimage.io.imread(args.input, plugin="tifffile").astype(np.float32)
stack_MRC = mrcfile.open(file_path)
noisy = stack_MRC.data

In [ ]:
Z_dim = noisy.shape[0]
Z2 = Z_dim//2

In [ ]:
def get_gaussian_kernel(sigma=1):
    number_of_coeffs = 3
    number_of_zeros = 0
    while number_of_zeros < 2 :
        delta = np.zeros(number_of_coeffs)
        delta[delta.size//2] = 1
        coeffs = scipy.ndimage.gaussian_filter1d(delta, sigma=sigma)
        number_of_zeros = coeffs.size - np.count_nonzero(coeffs)
        number_of_coeffs += 1
    return coeffs[1:-1]

std_dev = 2.0
sigma = np.array([std_dev, std_dev, std_dev])
kernel = [None]*3
kernel[0] = get_gaussian_kernel(sigma[0])
kernel[1] = get_gaussian_kernel(sigma[1])
kernel[2] = get_gaussian_kernel(sigma[2])

In [ ]:
denoiser = GF(logger)
denoised = denoiser.filter(noisy, kernel)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(denoised[Z2], cmap="gray")
axes[1].imshow(noisy[Z2], cmap="gray")
plt.show()

In [ ]:
with mrcfile.new(args.output1, overwrite=True) as mrc:
    mrc.set_data(denoised.astype(np.float32))
    mrc.data

In [ ]:
def read_MRC(file_path):
    return mrcfile.read(file_path)

In [ ]:
denoised = read_MRC(args.output1)

In [ ]:
figure(figsize=(32, 32))
plt.subplot(1, 3, 1)
plt.title("original")
imgplot = plt.imshow(noisy[7][::-1, :], cmap="gray")
plt.subplot(1, 3, 2)
plt.title("FlowDenoising")
plt.imshow(denoised[7][::-1, :], cmap="gray")
plt.subplot(1, 3, 3)
plt.title("difference")
plt.imshow(noisy[7][::-1, :] - denoised[7][::-1, :], cmap="gray")

In [ ]:
from matplotlib.pyplot import figure
figure(figsize=(16, 16))
slice_idx = denoised.shape[0]//2
plt.imshow(denoised[slice_idx, 0:400, 400:800], cmap="gray")
plt.savefig(args.output2, bbox_inches='tight')

In [ ]:
plt.close()

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        uploaded_file_id = handler.upload(
            local_file_path=output,
            drive_file_name=output,
            drive_folder_id=DRIVE_TOMOGRADENOISING_TMP)